In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


In [2]:
df=pd.read_csv("/content/drive/MyDrive/Capstone 2025/Panic_Attack_Extra_Feature_Version.csv")

In [3]:
# Feature Engineering
# 1. Symptom Severity Score
df['Symptom_Score'] = df['Sweating'] + df['Shortness_of_Breath'] + df['Dizziness'] + df['Chest_Pain'] + df['Trembling']



In [4]:

# 2. Lifestyle Impact Score (higher score means healthier lifestyle)
df['Lifestyle_Score'] = (
    df['Exercise_Frequency'] * 0.3 +  # Higher exercise is better
    (df['Sleep_Hours'] - 6) * 0.3 +   # More sleep (above 6 hours) is better
    (4 - df['Caffeine_Intake']) * 0.15 +  # Less caffeine is better
    (10 - df['Alcohol_Consumption']) * 0.15 +  # Less alcohol is better
    (1 - df['Smoking']) * 0.1  # Non-smoking is better
)


In [5]:

# 3. Recovery Time Estimation (estimated recovery in minutes)
df['Est_Recovery_Time'] = (
    df['Duration_Minutes'] * 0.5 +
    df['Symptom_Score'] * 5 +
    (df['Heart_Rate'] - 80) * 0.2
)

In [6]:
# 4. Stress Vulnerability Index
df['Stress_Vulnerability'] = (
    df['Panic_Attack_Frequency'] * 0.4 +
    (5 - df['Lifestyle_Score']) * 0.4 +
    df['Trigger'] * 0.2
)


In [7]:
# 5. Treatment Effectiveness
df['Has_Treatment'] = np.where((df['Medication'] == 1) | (df['Therapy'] == 1), 1, 0)
df['Combined_Treatment'] = np.where((df['Medication'] == 1) & (df['Therapy'] == 1), 1, 0)



In [8]:
# Visualization 1: Distribution of Panic Attack Frequency by Age Group
plt.figure(figsize=(10, 6))
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 30, 45, 60, 75], labels=['18-30', '31-45', '46-60', '61-75'])
sns.boxplot(x='Age_Group', y='Panic_Attack_Frequency', data=df)
plt.title('Distribution of Panic Attack Frequency by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Panic Attack Frequency')
plt.tight_layout()
plt.savefig('panic_attack_by_age.png')
plt.close()


In [9]:
# Visualization 2: Impact of Lifestyle Score on Symptom Severity
plt.figure(figsize=(12, 6))
sns.scatterplot(x='Lifestyle_Score', y='Symptom_Score', hue='Gender', data=df, alpha=0.6)
plt.title('Impact of Lifestyle on Symptom Severity')
plt.xlabel('Lifestyle Score (Higher is Healthier)')
plt.ylabel('Symptom Severity Score')
plt.axhline(y=df['Symptom_Score'].mean(), color='red', linestyle='--', label='Avg Symptom Score')
plt.axvline(x=df['Lifestyle_Score'].mean(), color='blue', linestyle='--', label='Avg Lifestyle Score')
plt.legend(title='Gender', labels=['Male', 'Female'])
plt.tight_layout()
plt.savefig('lifestyle_vs_symptoms.png')
plt.close()


In [10]:
# Visualization 3: Treatment Effectiveness
treatment_groups = df.groupby('Combined_Treatment')['Panic_Attack_Frequency'].mean().reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(x='Combined_Treatment', y='Panic_Attack_Frequency', data=treatment_groups)
plt.title('Effect of Combined Treatment on Panic Attack Frequency')
plt.xlabel('Combined Treatment (Medication + Therapy)')
plt.ylabel('Average Panic Attack Frequency')
plt.xticks([0, 1], ['No Combined Treatment', 'Combined Treatment'])
plt.tight_layout()
plt.savefig('treatment_effectiveness.png')
plt.close()


In [11]:
# Visualization 4: Heart Rate vs Duration with Symptom Score
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df['Heart_Rate'], df['Duration_Minutes'],
                     c=df['Symptom_Score'], cmap='viridis',
                     alpha=0.6, s=50)
plt.colorbar(scatter, label='Symptom Score')
plt.title('Relationship between Heart Rate, Attack Duration, and Symptom Severity')
plt.xlabel('Heart Rate (bpm)')
plt.ylabel('Duration (minutes)')
plt.tight_layout()
plt.savefig('heart_rate_duration_symptoms.png')
plt.close()



In [12]:
# Visualization 5: Sleep Quality vs Recovery Time
plt.figure(figsize=(10, 6))
sns.boxplot(x=pd.cut(df['Sleep_Hours'], bins=[4, 6, 7, 8, 10], labels=['4-6', '6-7', '7-8', '8+']))
plt.title('Impact of Sleep Hours on Estimated Recovery Time')
plt.xlabel('Sleep Hours')
plt.ylabel('Estimated Recovery Time (minutes)')
plt.tight_layout()
plt.savefig('sleep_vs_recovery.png')
plt.close()


In [13]:
# Visualization 6: Clustering Analysis
# Select relevant features for clustering
cluster_features = ['Panic_Attack_Frequency', 'Heart_Rate', 'Symptom_Score', 'Lifestyle_Score']
X = df[cluster_features].copy()



In [14]:
# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Determine optimal number of clusters using the elbow method
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Plot the elbow method
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, 'bo-')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.tight_layout()
plt.savefig('elbow_method.png')
plt.close()

# Choose k=3 for demonstration
kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Plot the clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(x='Lifestyle_Score', y='Panic_Attack_Frequency', hue='Cluster',
                palette='viridis', data=df, s=60, alpha=0.7)
plt.title('Clusters by Lifestyle Score and Panic Attack Frequency')
plt.xlabel('Lifestyle Score (Higher is Healthier)')
plt.ylabel('Panic Attack Frequency')
plt.tight_layout()
plt.savefig('lifestyle_clusters.png')
plt.close()

# Analyze clusters
cluster_analysis = df.groupby('Cluster')[cluster_features + ['Age', 'Gender', 'Sleep_Hours']].mean()
print("Cluster Analysis:")
print(cluster_analysis)


Cluster Analysis:
         Panic_Attack_Frequency  Heart_Rate  Symptom_Score  Lifestyle_Score  \
Cluster                                                                       
0                      6.457672  101.275132       2.835979         2.207989   
1                      5.451691  142.695652       2.314010         1.939710   
2                      1.463235  115.208333       3.061275         2.340588   

               Age    Gender  Sleep_Hours  
Cluster                                    
0        40.997354  0.648148     6.507672  
1        41.038647  0.664251     6.271256  
2        41.357843  0.600490     6.670833  


In [15]:
# Correlation analysis
plt.figure(figsize=(14, 10))
correlation_cols = ['Age', 'Panic_Attack_Frequency', 'Duration_Minutes', 'Heart_Rate',
                    'Symptom_Score', 'Caffeine_Intake', 'Exercise_Frequency',
                    'Sleep_Hours', 'Alcohol_Consumption', 'Lifestyle_Score','Est_Recovery_Time', 'Stress_Vulnerability']
corr = df[correlation_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', vmin=-1, vmax=1, center=0,
            square=True, linewidths=.5, annot=True, fmt='.2f', cbar_kws={'shrink': .5})
plt.title('Correlation Matrix of Key Features')
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.close()


In [16]:
# Summary Statistics for Key Derived Features
summary_stats = df[['Symptom_Score', 'Lifestyle_Score', 'Est_Recovery_Time',
                   'Stress_Vulnerability']].describe()
print("Summary Statistics for Derived Features:")
print(summary_stats)



Summary Statistics for Derived Features:
       Symptom_Score  Lifestyle_Score  Est_Recovery_Time  Stress_Vulnerability
count    1200.000000      1200.000000        1200.000000           1200.000000
mean        2.732500         2.160517          33.919250              3.397460
std         1.103158         0.882088           9.044322              1.236527
min         0.000000        -0.240000           7.000000              0.536000
25%         2.000000         1.537500          27.975000              2.388000
50%         3.000000         2.140000          33.750000              3.420000
75%         4.000000         2.800000          40.100000              4.389000
max         5.000000         4.600000          59.600000              6.512000


In [17]:
# Regression Analysis: Impact of Lifestyle on Panic Attack Frequency
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = df[['Lifestyle_Score', 'Sleep_Hours', 'Exercise_Frequency', 'Caffeine_Intake']]
y = df['Panic_Attack_Frequency']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

print("\nRegression Coefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}")



Regression Coefficients:
Lifestyle_Score: 0.2360
Sleep_Hours: -0.0219
Exercise_Frequency: -0.0072
Caffeine_Intake: -0.0035
Intercept: 3.9604


In [18]:
# Feature Importance Analysis
plt.figure(figsize=(10, 6))
features = X.columns
importances = abs(model.coef_)
indices = np.argsort(importances)

plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.title('Feature Importance for Predicting Panic Attack Frequency')
plt.xlabel('Absolute Coefficient Value')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from xgboost import XGBRegressor, XGBClassifier
import warnings
warnings.filterwarnings('ignore')


In [20]:
# Create Age Groups
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 30, 45, 60, 75], labels=[0, 1, 2, 3])

# Create a target variable for classification: High Frequency Attacks (≥10 per month)
df['High_Frequency'] = np.where(df['Panic_Attack_Frequency'] >= 10, 1, 0)



In [21]:
# For regression, we'll predict the Panic_Attack_Frequency
# For classification, we'll predict the High_Frequency


In [22]:
# Define features
categorical_features = ['Gender', 'Trigger', 'Medical_History', 'Age_Group']
numerical_features = [
    'Age', 'Heart_Rate', 'Sweating', 'Shortness_of_Breath',
    'Dizziness', 'Chest_Pain', 'Trembling', 'Medication',
    'Caffeine_Intake', 'Exercise_Frequency', 'Sleep_Hours',
    'Alcohol_Consumption', 'Smoking', 'Therapy',
    'Symptom_Score', 'Lifestyle_Score', 'Stress_Vulnerability',
    'Has_Treatment', 'Combined_Treatment'
]


In [23]:
# Preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])


In [24]:
# Define the model evaluation function
def evaluate_regression_model(model, X_train, X_test, y_train, y_test, model_name):
    # Train model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Feature importance (handle different model types)
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        importances = [0] * len(X_train.columns)  # Placeholder for models without feature_importances_

    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2,
        'Importance': importances
    }

def evaluate_classification_model(model, X_train, X_test, y_train, y_test, model_name):
    # Train model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    # Get classification report
    report = classification_report(y_test, y_pred, output_dict=True)

    # Feature importance (handle different model types)
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        importances = [0] * len(X_train.columns)  # Placeholder

   # Get the unique classes
    classes = sorted(np.unique(y_test))
    positive_class = classes[-1]  # Assuming last class is positive

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': report[str(positive_class)]['precision'],
        'Recall': report[str(positive_class)]['recall'],
        'F1': report[str(positive_class)]['f1-score'],
    }



In [25]:
# Split the data for regression
X_reg = df[numerical_features + categorical_features]
y_reg = df['Panic_Attack_Frequency']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)

# Split the data for classification
X_class = df[numerical_features + categorical_features]
y_class = df['High_Frequency']
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(X_class, y_class, test_size=0.25, random_state=42)



In [26]:
# Define models for regression
rf_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

xgb_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])



In [27]:
# Define models for classification
rf_class = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

xgb_class = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42))
])


In [28]:
# Evaluate regression models
reg_models = [
    (rf_reg, "Random Forest"),
    (xgb_reg, "XGBoost")]

In [29]:
reg_results = []
for model, name in reg_models:
    result = evaluate_regression_model(model, X_train_reg, X_test_reg, y_train_reg, y_test_reg, name)
    reg_results.append(result)

# Evaluate classification models
class_models = [
    (rf_class, "Random Forest"),
    (xgb_class, "XGBoost")]

In [30]:
class_results = []
for model, name in class_models:
    result = evaluate_classification_model(model, X_train_class, X_test_class, y_train_class, y_test_class, name)
    class_results.append(result)

# Create comparison tables
reg_df = pd.DataFrame(reg_results)[['Model', 'MSE', 'RMSE', 'R²']]
class_df = pd.DataFrame(class_results)[['Model', 'Accuracy', 'Precision', 'Recall', 'F1']]

print("Regression Model Comparison:")
print(reg_df)
print("\nClassification Model Comparison:")
print(class_df)



Regression Model Comparison:
           Model       MSE      RMSE        R²
0  Random Forest  0.230718  0.480331  0.968543
1        XGBoost  0.109937  0.331568  0.985011

Classification Model Comparison:
           Model  Accuracy  Precision  Recall   F1
0  Random Forest       1.0        1.0     1.0  1.0
1        XGBoost       1.0        1.0     1.0  1.0


In [31]:
# Visualize regression results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.barplot(x='Model', y='RMSE', data=reg_df)
plt.title('Regression Models: RMSE (lower is better)')
plt.subplot(1, 2, 2)
sns.barplot(x='Model', y='R²', data=reg_df)
plt.title('Regression Models: R² (higher is better)')
plt.tight_layout()
plt.savefig('regression_comparison.png')
plt.close()

# Visualize classification results
plt.figure(figsize=(14, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
for i, metric in enumerate(metrics):
    plt.subplot(1, 4, i+1)
    sns.barplot(x='Model', y=metric, data=class_df)
    plt.title(f'Classification Models: {metric}')
    plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('classification_comparison.png')
plt.close()


In [32]:
# Cross-validation for regression models
cv_scores = {}
for model, name in reg_models:
    scores = cross_val_score(model, X_reg, y_reg, cv=5, scoring='neg_mean_squared_error')
    cv_scores[name] = -scores.mean()

# Cross-validation for classification models
cv_scores_class = {}
for model, name in class_models:
    scores = cross_val_score(model, X_class, y_class, cv=5, scoring='accuracy')
    cv_scores_class[name] = scores.mean()


In [33]:
# Visualize cross-validation results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.bar(cv_scores.keys(), cv_scores.values())
plt.title('5-Fold CV: MSE for Regression Models (lower is better)')
plt.ylabel('Mean Squared Error')

plt.subplot(1, 2, 2)
plt.bar(cv_scores_class.keys(), cv_scores_class.values())
plt.title('5-Fold CV: Accuracy for Classification Models (higher is better)')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

plt.tight_layout()
plt.savefig('cross_validation_results.png')
plt.close()

# Find the best model for both tasks
best_reg_model = min(cv_scores.items(), key=lambda x: x[1])[0]
best_class_model = max(cv_scores_class.items(), key=lambda x: x[1])[0]

print(f"\nBest regression model based on cross-validation: {best_reg_model}")
print(f"Best classification model based on cross-validation: {best_class_model}")




Best regression model based on cross-validation: XGBoost
Best classification model based on cross-validation: Random Forest


In [34]:
# Hyperparameter tuning for the best models
if best_reg_model == "Random Forest":
    param_grid_reg = {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5, 10]
    }
    best_reg_pipeline = rf_reg
elif best_reg_model == "XGBoost":
    param_grid_reg = {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.1, 0.2],
        'model__max_depth': [3, 5, 7]
    }
    best_reg_pipeline = xgb_reg

if best_class_model == "Random Forest":
    param_grid_class = {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5, 10]
    }
    best_class_pipeline = rf_class
elif best_class_model == "XGBoost":
    param_grid_class = {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.1, 0.2],
        'model__max_depth': [3, 5, 7]
    }
    best_class_pipeline = xgb_class

# Perform grid search for regression
grid_search_reg = GridSearchCV(best_reg_pipeline, param_grid_reg, cv=3, scoring='neg_mean_squared_error')
grid_search_reg.fit(X_reg, y_reg)

# Perform grid search for classification
grid_search_class = GridSearchCV(best_class_pipeline, param_grid_class, cv=3, scoring='accuracy')
grid_search_class.fit(X_class, y_class)

print(f"\nBest parameters for {best_reg_model} (regression):")
print(grid_search_reg.best_params_)
print(f"Best cross-validation score: {-grid_search_reg.best_score_:.4f} MSE")

print(f"\nBest parameters for {best_class_model} (classification):")
print(grid_search_class.best_params_)
print(f"Best cross-validation score: {grid_search_class.best_score_:.4f} accuracy")

# Feature importance for the best tuned models (using the best model from GridSearchCV)
best_reg_model_tuned = grid_search_reg.best_estimator_
best_class_model_tuned = grid_search_class.best_estimator_

# For regression: Get feature names after preprocessing
# This is complex due to preprocessing transformations, so we'll use a simpler approach
# Train a model directly without pipeline to get feature importance
if best_reg_model == "Random Forest":
    direct_model = RandomForestRegressor(**{k.replace('model__', ''): v for k, v in grid_search_reg.best_params_.items()})
elif best_reg_model == "XGBoost":
    direct_model = XGBRegressor(**{k.replace('model__', ''): v for k, v in grid_search_reg.best_params_.items()})

# One-hot encode categorical features for direct model
X_encoded = pd.get_dummies(X_reg, columns=categorical_features)
direct_model.fit(X_encoded, y_reg)

# Get feature importance
if hasattr(direct_model, 'feature_importances_'):
    feature_importance = direct_model.feature_importances_
    feature_names = X_encoded.columns

    # Create dataframe for feature importance
    feat_imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': feature_importance
    }).sort_values('Importance', ascending=False).head(15)

    # Plot feature importance
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=feat_imp_df)
    plt.title(f'Top 15 Feature Importances for {best_reg_model} (Regression)')
    plt.tight_layout()
    plt.savefig('feature_importance_regression.png')
    plt.close()

    print("\nTop 10 Features for Predicting Panic Attack Frequency:")
    print(feat_imp_df.head(10))

# For classification: similar approach
if best_class_model == "Random Forest":
    direct_model_class = RandomForestClassifier(**{k.replace('model__', ''): v for k, v in grid_search_class.best_params_.items()})
elif best_class_model == "XGBoost":
    direct_model_class = XGBClassifier(**{k.replace('model__', ''): v for k, v in grid_search_class.best_params_.items()})

direct_model_class.fit(X_encoded, y_class)

# Get feature importance for classification
if hasattr(direct_model_class, 'feature_importances_'):
    feature_importance_class = direct_model_class.feature_importances_

    # Create dataframe for feature importance
    feat_imp_df_class = pd.DataFrame({
        'Feature': feature_names,
        'Importance': feature_importance_class
    }).sort_values('Importance', ascending=False).head(15)

    # Plot feature importance
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=feat_imp_df_class)
    plt.title(f'Top 15 Feature Importances for {best_class_model} (Classification)')
    plt.tight_layout()
    plt.savefig('feature_importance_classification.png')
    plt.close()

    print("\nTop 10 Features for Predicting High Frequency Attacks:")
    print(feat_imp_df_class.head(10))

# Generate confusion matrix for the best classification model
best_class_model_tuned.fit(X_train_class, y_train_class)
y_pred_class = best_class_model_tuned.predict(X_test_class)
cm = confusion_matrix(y_test_class, y_pred_class)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix for {best_class_model}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix.png')
plt.close()

# Summary document
summary = f"""
# Machine Learning Model Comparison for Anxiety Attack Prediction

## Regression Task: Predicting Panic Attack Frequency

| Model | MSE | RMSE | R² |
|-------|-----|------|---|
{' | '.join(reg_df.iloc[0].values.astype(str))}
{' | '.join(reg_df.iloc[1].values.astype(str))}

Best model: {best_reg_model} with CV MSE of {min(cv_scores.values()):.4f}

## Classification Task: Predicting High Frequency Attacks

| Model | Accuracy | Precision | Recall | F1 |
|-------|----------|-----------|--------|---|
{' | '.join(class_df.iloc[0].values.astype(str))}
{' | '.join(class_df.iloc[1].values.astype(str))}

Best model: {best_class_model} with CV accuracy of {max(cv_scores_class.values()):.4f}
"""


Best parameters for XGBoost (regression):
{'model__learning_rate': 0.2, 'model__max_depth': 3, 'model__n_estimators': 200}
Best cross-validation score: 0.0555 MSE

Best parameters for Random Forest (classification):
{'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 50}
Best cross-validation score: 1.0000 accuracy

Top 10 Features for Predicting Panic Attack Frequency:
                 Feature  Importance
16  Stress_Vulnerability    0.517804
27             Trigger_5    0.115581
22             Trigger_0    0.108532
26             Trigger_4    0.074206
23             Trigger_1    0.073237
15       Lifestyle_Score    0.061686
24             Trigger_2    0.024895
25             Trigger_3    0.004612
11   Alcohol_Consumption    0.002616
3    Shortness_of_Breath    0.002506

Top 10 Features for Predicting High Frequency Attacks:
               Feature  Importance
0                  Age         0.0
1           Heart_Rate         0.0
2             Sweating       